<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/saude/MortalidadeInfantil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mortalidade infantil — taxa por mil nascidos vivos

Este notebook é o motor de coleta e publicação da série `mortalidade-infantil-anual`.

- **Fonte:** Ministério da Saúde/Secretaria de Vigilância em Saúde e Ambiente (MS/SVSA) —
  microdados do Sistema de Informações sobre Mortalidade (SIM) e do Sistema de Informações
  sobre Nascidos Vivos (SINASC), distribuídos pelo DATASUS via FTP (`ftp.datasus.gov.br`).
  - Numerador: óbitos de menores de 1 ano de idade — arquivo nacional `DOINF{AA}.dbc` em
    `dissemin/publicos/SIM/CID10/DOFET/` (um por ano).
  - Denominador: nascidos vivos — arquivo nacional `DNBR{AAAA}.dbc` em
    `dissemin/publicos/SINASC/1996_/Dados/DNRES/` (um por ano).
- **Definição:** número de óbitos de menores de 1 ano por mil nascidos vivos, no
  ano-calendário, por residência. É o cálculo **direto** a partir das contagens oficiais do
  SIM e do SINASC, **sem correção de sub-registro**.
- **Fórmula:** `taxa = 1000 × óbitos_de_menores_de_1_ano ÷ nascidos_vivos`, por ano.
- **Unidade:** por mil nascidos vivos (`unidade.tipo: numero`, `simbolo: por mil`,
  `casas: 1`); `eixo: por-mil`, `medida: nivel`, `granularidade: anual`.
- **Período publicado:** `1996-01-01` até o último ano com SIM e SINASC **finais**.
- **Arquivo gerado:** `indicadores/saude/mortalidade-infantil-anual.json`.

## O que este número é — e o que ele não é

- **É o cálculo direto, sem correção de sub-registro.** Por isso fica sistematicamente
  abaixo da taxa indireta do IBGE (série CD100, 2000–2016), que corrige o sub-registro a
  partir do Censo: em 2000 a diferença é de ~7,7 por mil e em 2016 de ~0,6 por mil. As
  publicações de referência usam a mesma construção direta, cada uma no seu recorte: 12,62
  em 2023 é o boletim de mortalidade materna e infantil da SES/RS (Brasil) e 12,6 em 2024 é
  a estimativa da Fundação Abrinq. O cálculo direto deste notebook dá 12,56 em 2024
  (30.020 óbitos ÷ 2.389.325 nascidos × 1000).
- **Não é a taxa neonatal nem a de menores de 5 anos** — só o primeiro ano de vida.
- **Os anos recentes são revisados.** O SIM e o SINASC consolidam as declarações com atraso:
  um ano só entra na série quando os dois sistemas o publicam como final. Reexecutar este
  notebook muda valores apenas se o DATASUS revisar a base; aí é revisão de fonte,
  registrada em `docs/metodologia.md`.

## Por que microdados e não o TabNet

A taxa pronta do TabNet do SIM/SINASC (`sim/cnv/inf10uf.def`) não está scriptável hoje: o
servidor devolve "Tabela de conversão não encontrada" ao tabular. Por isso o notebook
recalcula a taxa a partir dos **microdados** (as contagens oficiais): lê no FTP do DATASUS
o cabeçalho de `DOINF` (SIM) e `DNBR` (SINASC), onde está o total de registros de cada
arquivo, e aplica a fórmula acima. A fórmula fica visível e auditável.

O volume do SINASC é grande (~120 MB por ano comprimido) e o FTP do DATASUS entrega cerca
de 1,7 MB/s: baixar os 29 arquivos levava 35 min, acima do limite de execução do CI. Por
isso a contagem sai do **cabeçalho DBF**, que o `.dbc` guarda sem compressão nos primeiros
bytes — o total de registros está lá, é a contagem oficial do arquivo, e o resto (os
registros comprimidos) não precisa ser baixado. Cada leitura é uma conexão FTP curta e elas
rodam em paralelo.

In [1]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import ftplib
import json
import math
import os
import re
import struct
import time
from decimal import Decimal, ROUND_HALF_UP

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()
if os.environ.get("DADOS_RAIZ"):
    PASTA = RAIZ / "indicadores" / "saude"
    if not PASTA.is_dir():
        raise SystemExit(f"família não encontrada em {PASTA}")
else:
    PASTA = RAIZ
DESTINO = Path(os.environ.get("SAUDE_SAIDA", PASTA))


def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name


FTP_HOST = "ftp.datasus.gov.br"
# Diretórios dos microdados FINAIS (o ano só entra quando SIM e SINASC o publicam como final).
DIR_SIM = "/dissemin/publicos/SIM/CID10/DOFET"
DIR_SINASC = "/dissemin/publicos/SINASC/1996_/Dados/DNRES"
INICIO = 1996
# Conexões FTP simultâneas. Cada worker abre a própria conexão; oito cabem na janela do CI.
CONEXOES = 8

In [2]:
def listar(caminho):
    """Lista um diretório do FTP DATASUS, com reconexão e espera crescente."""
    for tentativa in range(1, 6):
        try:
            ftp = ftplib.FTP(FTP_HOST, timeout=60)
            ftp.login()
            itens = []
            ftp.retrlines("NLST " + caminho, itens.append)
            ftp.quit()
            return [Path(item).name for item in itens]
        except Exception:
            if tentativa == 5:
                raise
            time.sleep(2 ** tentativa)


class Suficiente(Exception):
    """Sentinela interna: o prefixo baixado já traz o cabeçalho DBF completo."""


def _contar_cabecalho(caminho):
    """Conta os registros de um .dbc lendo só o cabeçalho DBF, sem baixar o arquivo todo.

    O .dbc do DATASUS é um DBF com o cabeçalho **sem compressão** nos primeiros bytes: o
    total de registros está nos bytes 4–7 (little-endian) e o tamanho do cabeçalho nos
    bytes 8–9. Só os registros (o resto do arquivo) vêm comprimidos — e não é preciso
    baixá-los para contar: o cabeçalho é a contagem oficial que o DATASUS gravou, idêntica
    à varredura do arquivo inteiro (conferida nos 29 anos em 2026-11-05). Baixar os ~120 MB
    por ano do SINASC levava 35 min, acima do limite de execução do CI.
    """
    pedacos = []
    necessario = [10]  # os bytes 8–9 dizem o tamanho total do cabeçalho

    def acumular(bloco):
        pedacos.append(bloco)
        if sum(map(len, pedacos)) < necessario[0]:
            return
        dados = b"".join(pedacos)
        if necessario[0] == 10:
            necessario[0] = max(10, struct.unpack("<H", dados[8:10])[0])
            if len(dados) < necessario[0]:
                return
        raise Suficiente

    ftp = ftplib.FTP(FTP_HOST, timeout=90)
    ftp.login()
    try:
        ftp.retrbinary("RETR " + caminho, acumular, blocksize=8192)
    except Suficiente:
        pass
    finally:
        ftp.close()
    dados = b"".join(pedacos)
    if len(dados) < necessario[0]:
        raise ValueError(f"cabeçalho incompleto em {caminho}")
    return struct.unpack("<I", dados[4:8])[0]


def contar_registros(caminho):
    """Conta os registros de um .dbc, com reconexão e espera crescente."""
    for tentativa in range(1, 6):
        try:
            return _contar_cabecalho(caminho)
        except Exception:
            if tentativa == 5:
                raise
            time.sleep(2 ** tentativa)


def ano_do_doinf(nome):
    m = re.fullmatch(r"DOINF(\d{2})\.dbc", nome, re.IGNORECASE)
    if not m:
        return None
    aa = int(m.group(1))
    # Pivô de dois dígitos: 96–99 → 1996–1999 e 00–89 → 2000–2089. O corte em 90 (e não em
    # 25) não depende do ano corrente nem transforma DOINF26 em 1926.
    return 2000 + aa if aa < 90 else 1900 + aa


def ano_do_dnbr(nome):
    m = re.fullmatch(r"DNBR(\d{4})\.dbc", nome, re.IGNORECASE)
    return int(m.group(1)) if m else None


obitos_por_ano = {}
for nome in listar(DIR_SIM):
    ano = ano_do_doinf(nome)
    if ano is not None and ano >= INICIO:
        obitos_por_ano[ano] = f"{DIR_SIM}/{nome}"

nascidos_por_ano = {}
for nome in listar(DIR_SINASC):
    ano = ano_do_dnbr(nome)
    if ano is not None and ano >= INICIO:
        nascidos_por_ano[ano] = f"{DIR_SINASC}/{nome}"

# Um ano só entra quando o SIM e o SINASC o publicam como final; o ano corrente (ainda
# preliminar) e anos com só um dos lados final ficam de fora. A exclusão é impressa: um
# arquivo novo que "desaparece" da série não pode passar em silêncio.
anos = sorted(set(obitos_por_ano) & set(nascidos_por_ano))
excluidos = sorted(set(obitos_por_ano) ^ set(nascidos_por_ano))
print(f"anos com SIM e SINASC finais: {anos[0]}–{anos[-1]} ({len(anos)} anos)")
print(f"anos excluídos (só um dos sistemas finalizado): {excluidos}")

anos com SIM e SINASC finais: 1996–2024 (29 anos)
anos excluídos (só um dos sistemas finalizado): [2025]


In [3]:
def contar_ano(ano):
    """Conta os dois microdados do ano (worker: uma conexão FTP curta por arquivo)."""
    return (
        ano,
        contar_registros(obitos_por_ano[ano]),
        contar_registros(nascidos_por_ano[ano]),
    )


# Cada contagem é uma conexão FTP curta (só o cabeçalho, alguns KB): CONEXOES rodam em
# paralelo. O ``map`` devolve na ordem dos anos, então a saída não depende de quem termina
# primeiro.
obitos = {}
nascidos = {}
with ThreadPoolExecutor(max_workers=CONEXOES) as executor:
    for ano, mortos, nascidos_no_ano in executor.map(contar_ano, anos):
        obitos[ano] = mortos
        nascidos[ano] = nascidos_no_ano
        print(f"{ano}: {obitos[ano]:,} óbitos <1 ano, {nascidos[ano]:,} nascidos vivos")


def arredondar(valor):
    """Arredonda para 2 casas com meio para cima (convenção comercial)."""
    return float(Decimal(repr(valor)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))


# Fórmula: taxa = 1000 × óbitos de menores de 1 ano ÷ nascidos vivos (por residência).
taxa = {ano: arredondar(1000.0 * obitos[ano] / nascidos[ano]) for ano in anos}

print()
print(f"{'ano':>4} {'óbitos':>9} {'nascidos':>11} {'taxa':>6}")
for ano in anos:
    print(f"{ano:>4} {obitos[ano]:>9,} {nascidos[ano]:>11,} {taxa[ano]:>6.2f}")

1996: 75,024 óbitos <1 ano, 2,945,425 nascidos vivos
1997: 71,400 óbitos <1 ano, 3,026,658 nascidos vivos


1998: 71,690 óbitos <1 ano, 3,148,037 nascidos vivos
1999: 69,345 óbitos <1 ano, 3,256,433 nascidos vivos
2000: 68,199 óbitos <1 ano, 3,206,761 nascidos vivos
2001: 61,943 óbitos <1 ano, 3,115,474 nascidos vivos
2002: 58,916 óbitos <1 ano, 3,059,402 nascidos vivos
2003: 57,540 óbitos <1 ano, 3,038,251 nascidos vivos


2004: 54,183 óbitos <1 ano, 3,026,548 nascidos vivos
2005: 51,544 óbitos <1 ano, 3,035,096 nascidos vivos
2006: 48,332 óbitos <1 ano, 2,944,928 nascidos vivos
2007: 45,370 óbitos <1 ano, 2,891,328 nascidos vivos
2008: 44,100 óbitos <1 ano, 2,934,828 nascidos vivos
2009: 42,661 óbitos <1 ano, 2,881,581 nascidos vivos
2010: 39,874 óbitos <1 ano, 2,861,868 nascidos vivos
2011: 39,716 óbitos <1 ano, 2,913,160 nascidos vivos
2012: 39,123 óbitos <1 ano, 2,905,789 nascidos vivos
2013: 38,966 óbitos <1 ano, 2,904,027 nascidos vivos
2014: 38,432 óbitos <1 ano, 2,979,259 nascidos vivos
2015: 37,501 óbitos <1 ano, 3,017,668 nascidos vivos
2016: 36,350 óbitos <1 ano, 2,857,800 nascidos vivos
2017: 36,223 óbitos <1 ano, 2,923,535 nascidos vivos
2018: 35,864 óbitos <1 ano, 2,944,932 nascidos vivos
2019: 35,293 óbitos <1 ano, 2,849,146 nascidos vivos
2020: 31,439 óbitos <1 ano, 2,730,145 nascidos vivos
2021: 31,856 óbitos <1 ano, 2,677,101 nascidos vivos
2022: 32,257 óbitos <1 ano, 2,561,922 nascidos

In [4]:
def pontos(serie):
    return [
        {"data": f"{ano}-01-01", "valor": valor}
        for ano, valor in sorted(serie.items())
    ]


FONTE = (
    "MS/SVSA — SIM (óbitos de menores de 1 ano) e SINASC (nascidos vivos), microdados do "
    "DATASUS; taxa por mil nascidos vivos calculada pelo site: 1000 × óbitos ÷ nascidos vivos"
)

# Sem urlFonte: o recorte é um caminho FTP, e o contrato (C7.1) só aceita https:// em
# urlFonte. Os caminhos exatos dos arquivos ficam no markdown e em docs/metodologia.md.
serie = {
    "versao": 1,
    "id": "mortalidade-infantil-anual",
    "unidade": {"tipo": "numero", "simbolo": "por mil", "casas": 1},
    "eixo": "por-mil",
    "medida": "nivel",
    "granularidade": "anual",
    "fonte": FONTE,
    "dados": pontos(taxa),
}

DESTINO.mkdir(parents=True, exist_ok=True)
arquivo = DESTINO / "mortalidade-infantil-anual.json"
with arquivo.open("w", encoding="utf-8") as saida:
    json.dump(serie, saida, ensure_ascii=False, indent=2)
    saida.write("\n")
print(f"{len(serie['dados'])} pontos escritos em {relativo(arquivo)}")

29 pontos escritos em indicadores/saude/mortalidade-infantil-anual.json


In [5]:
CHAVES_RAIZ = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "dados",
}

assert set(serie) == CHAVES_RAIZ, sorted(set(serie) ^ CHAVES_RAIZ)
assert serie["versao"] == 1
assert serie["id"] == "mortalidade-infantil-anual"
assert serie["unidade"] == {"tipo": "numero", "simbolo": "por mil", "casas": 1}
assert serie["eixo"] == "por-mil"
assert serie["medida"] == "nivel"
assert serie["granularidade"] == "anual"
assert isinstance(serie["fonte"], str) and serie["fonte"]
# C7.1: urlFonte é opcional e, quando existe, precisa ser https://. O recorte desta série é
# um caminho FTP (ftp:// é proibido), então não há urlFonte — os caminhos ficam no markdown.
assert "urlFonte" not in serie, "urlFonte só entra com https://; o recorte aqui é ftp://"

dados = serie["dados"]
assert dados, "série vazia"
datas = [ponto["data"] for ponto in dados]
anos_lista = [int(data[:4]) for data in datas]

# Série anual: todo ponto em AAAA-01-01 (C15). A paridade bienal do C6 (paridade do
# primeiro ponto) vale para granularidade "bienal", não para esta série.
assert datas[0] == "1996-01-01", datas[0]
assert all(data.endswith("-01-01") for data in datas), "dia diferente de 01"
assert all(a < b for a, b in zip(datas, datas[1:])), "datas fora de ordem"
assert anos_lista == list(range(anos_lista[0], anos_lista[-1] + 1)), "ano faltando ou repetido"

for ponto in dados:
    assert set(ponto) == {"data", "valor"}, sorted(ponto)
    assert not isinstance(ponto["valor"], bool), ponto
    assert isinstance(ponto["valor"], (int, float)), ponto
    assert math.isfinite(ponto["valor"]), ponto

# Fórmula: cada ponto é 1000 × óbitos ÷ nascidos do mesmo ano.
for ponto in dados:
    ano = int(ponto["data"][:4])
    assert ponto["valor"] == arredondar(1000.0 * obitos[ano] / nascidos[ano]), ponto

# Confere a gravação em disco.
with (DESTINO / "mortalidade-infantil-anual.json").open(encoding="utf-8") as arquivo:
    assert json.load(arquivo) == serie, "arquivo gravado difere do objeto"


def valor(ano):
    return {int(p["data"][:4]): p["valor"] for p in dados}[ano]


# Cobertura: o começo é fixo; o fim é o último ano que a fonte fechou, então só pode
# crescer. As âncoras de valor abaixo falham numa revisão de fonte — se acontecer, confira
# os microdados e registre a revisão em docs/metodologia.md antes de mexer no número.
assert valor(1996) == 25.47, "1996 25.47 — revisão de fonte? veja docs/metodologia.md"
assert int(dados[-1]["data"][:4]) >= 2024, (
    f"último ano {dados[-1]['data']}: a série publicada ia até 2024; "
    "se perdeu um ano, veja docs/metodologia.md"
)
assert len(dados) >= 29, (
    f"{len(dados)} pontos: a série publicada tinha 29 (1996–2024); se a fonte fechou "
    "mais um ano, atualize o piso; se perdeu um ano, veja docs/metodologia.md"
)

# Números citados na metodologia e na evidência (taxa direta SIM/Sinasc).
assert valor(2000) == 21.27, "2000 21.27 — revisão de fonte? veja docs/metodologia.md"
assert valor(2016) == 12.72, "2016 12.72 — revisão de fonte? veja docs/metodologia.md"
assert valor(2020) == 11.52, "2020 11.52 — revisão de fonte? veja docs/metodologia.md"
assert valor(2023) == 12.62, "2023 12.62 — revisão de fonte? veja docs/metodologia.md"
assert valor(2024) == 12.56, "2024 12.56 — revisão de fonte? veja docs/metodologia.md"

print(
    f"ok: {len(dados)} pontos ({datas[0]} a {datas[-1]}); fórmula, dia 01, ordem, "
    "cobertura e contrato conferidos."
)

ok: 29 pontos (1996-01-01 a 2024-01-01); fórmula, dia 01, ordem, cobertura e contrato conferidos.
